In [38]:
pip install openpyxl


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /Users/rain/Desktop/DSP/COEQWAL_V3/.venv311/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [39]:
pip install pandas


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /Users/rain/Desktop/DSP/COEQWAL_V3/.venv311/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [40]:
pip install numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /Users/rain/Desktop/DSP/COEQWAL_V3/.venv311/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [41]:
pip install plotly


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /Users/rain/Desktop/DSP/COEQWAL_V3/.venv311/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [42]:
pip install matplotlib


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /Users/rain/Desktop/DSP/COEQWAL_V3/.venv311/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [43]:
pip install seaborn


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /Users/rain/Desktop/DSP/COEQWAL_V3/.venv311/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [44]:
pip install dash


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: /Users/rain/Desktop/DSP/COEQWAL_V3/.venv311/bin/python -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [45]:
import plotly.graph_objects as go

In [46]:
# Import standard libraries
import os
from contextlib import redirect_stdout

import sys
# append coeqwal packages to path
sys.path.append('./coeqwalpackage')

import numpy as np
import pandas as pd
import datetime as dt

In [47]:
# Import custom libraries
# Note: on my computer the next import doesn't work the first time I call it, why? If I re-run the cell, then it is ok. MUST DEBUG
from coeqwalpackage.metrics import *
import cqwlutils as cu
import plotting as pu
import re
import dash
from dash import dcc, html
from dash.dependencies import Input, Output

In [48]:
CtrlFile = 'CalSim3DataExtractionInitFile_v4.xlsx'
CtrlTab = 'Init'
ScenarioListFile, ScenarioListTab, ScenarioListPath, DVDssNamesOutPath, SVDssNamesOutPath, ScenarioIndicesOutPath, DssDirsOutPath, VarListPath, VarListFile, VarListTab, VarOutPath, DataOutPath, ConvertDataOutPath, ExtractionSubPath, DemandDeliverySubPath, ModelSubPath, GroupDataDirPath, ScenarioDir, DVDssMin, DVDssMax, SVDssMin, SVDssMax, NameMin, NameMax, DirMin, DirMax, IndexMin, IndexMax, StartMin, StartMax, EndMin, EndMax, VarMin, VarMax, DemandFilePath, DemandFileName, DemandFileTab, DemMin, DemMax, InflowOutSubPath, InflowFilePath, InflowFileName, InflowFileTab, InflowMin, InflowMax = cu.read_init_file(CtrlFile, CtrlTab)

In [49]:
df, dss_names = read_in_df(ConvertDataOutPath,DVDssNamesOutPath)

In [50]:
df = add_water_year_column(df)

In [51]:
metrics_path = GroupDataDirPath + "/metrics_output"
if not os.path.exists(metrics_path):
    os.makedirs(metrics_path)

plots_path = GroupDataDirPath + "/plots_output"
if not os.path.exists(plots_path):
    os.makedirs(plots_path)

output_data_path = "./Dashboard/data"
if not os.path.exists(output_data_path):
    os.makedirs(output_data_path)

output_filename = output_data_path + "/calsim_dashboard_df.csv"
output_group_data_filename = GroupDataDirPath + "/calsim_dashboard_df.csv"

In [52]:
drought_wys = [
    1924,1925,1926,1929,1930,1931,1932,1933,1934,
    1939,1944,1945,1947,1948,1949,1950,1955,1960,
    1961,1962,1964,1976,1977,1979,1981,1987,1988,
    1989,1990,1991,1992,1994,2001,2008,2009,2013,
    2014,2015,2020,2021
]

In [53]:
df.columns = ['_'.join(map(str, col)) for col in df.columns]

print(df.columns.tolist()[:20]) 

['WaterYear______', 'CALSIM_AWOANN_64_XADV_s0001_ANNUAL-APPLIED-WATER_1MON_L2020A_PER-AVER_TAF', 'CALSIM_AWOANN_72_XA1DV_s0001_ANNUAL-APPLIED-WATER_1MON_L2020A_PER-AVER_TAF', 'CALSIM_AWOANN_72_XA2DV_s0001_ANNUAL-APPLIED-WATER_1MON_L2020A_PER-AVER_TAF', 'CALSIM_AWOANN_72_XA3DV_s0001_ANNUAL-APPLIED-WATER_1MON_L2020A_PER-AVER_TAF', 'CALSIM_AWOANN_73_XADV_s0001_ANNUAL-APPLIED-WATER_1MON_L2020A_PER-AVER_TAF', 'CALSIM_BANKSEC_MAX14DAY_s0001_SALINITY-APPROX_1MON_L2020A_PER-AVER_UMHOS/CM', 'CALSIM_COREQSACDV_s0001_FLOW_1MON_L2020A_PER-AVER_CFS', 'CALSIM_CO_EC_MONTH_s0001_SALINITY_1MON_L2020A_PER-AVER_UMHOS/CM', 'CALSIM_C_AMR004_s0001_CHANNEL_1MON_L2020A_PER-AVER_CFS', 'CALSIM_C_AMR004_ADD_s0001_FLOW-ADDITIONAL-INSTREAM_1MON_L2020A_PER-AVER_CFS', 'CALSIM_C_AMR004_MIF_s0001_FLOW-MIN-INSTREAM_1MON_L2020A_PER-AVER_CFS', 'CALSIM_C_CAA003_s0001_CHANNEL_1MON_L2020A_PER-AVER_CFS', 'CALSIM_C_CAA003_CVP_s0001_FLOW-DELIVERY_1MON_L2020A_PER-AVER_CFS', 'CALSIM_C_CAA003_SWP_s0001_FLOW-DELIVERY_1MON_L2020A_P

In [54]:
original_columns = df.columns.tolist()

s_numbers = set()
for col in original_columns:
    matches = re.findall(r's\d{4,}', col)  
    s_numbers.update(matches) 

# Convert to a sorted list
s_numbers_list = sorted(s_numbers)

print(s_numbers_list)

['s0001', 's0002', 's0003', 's0004', 's0005', 's0006', 's0007', 's0008', 's0009', 's0010', 's0011', 's0012', 's0013', 's0014', 's0015', 's0016', 's0018', 's0019', 's0020', 's0021', 's0022', 's0023', 's0024', 's0025', 's0027', 's0029', 's0046']


### Time Series
Shows how the selected variable changes over time for each scenario.

### Monthly-of-Year
Displays the monthly average for a selected year.

### Single Exceedance
Shows the probability that a value will be equaled or exceeded.

### Annual Exceedance
Shows how often the selected month's total value exceeds a given threshold across all years. Each year’s data for the chosen month is summed (e.g., total flow in April each year), and the annual values are ranked from highest to lowest. From these ranks, exceedance probabilities are calculated to show how frequently high values occur.

### Month-of-Year Avg
Averages each calendar month across all years, optionally filtered by Water Year Type. Water Year Types classify each year based on how wet or dry it was. The scale ranges from 1 (wettest) to 5 (driest)

In [55]:
import re
import numpy as np
import pandas as pd
import plotly.graph_objs as go
import dash
from dash import dcc, html
from dash.dependencies import Input, Output, State
from dash.exceptions import PreventUpdate

DATA_PATH = "calsim_dashboard_df.csv"
VARIABLE_GROUPS_PATH = "variable_groupings.csv"
SCENARIO_GROUPS_PATH = "scenario_groupings.csv"

df = pd.read_csv(DATA_PATH, parse_dates=[0], index_col=0)
df.index = pd.to_datetime(df.index)

def read_csv_flexible(path):
    try:
        return pd.read_csv(path, encoding="utf-8")
    except Exception:
        return pd.read_csv(path, encoding="latin1")

var_groups_raw = read_csv_flexible(VARIABLE_GROUPS_PATH)
scen_groups_raw = read_csv_flexible(SCENARIO_GROUPS_PATH)

original_columns = list(map(str, df.columns))

pat_var = re.compile(r"CALSIM_(.*?)_s\d{4}", re.IGNORECASE)
pat_s = re.compile(r"s\d{4,}", re.IGNORECASE)

records = []
for col in original_columns:
    m = pat_var.search(col)
    var = m.group(1) if m else None
    s_list = pat_s.findall(col)
    scen = s_list[0] if s_list else None
    unit = col.split("_")[-1] if "_" in col else ""
    records.append({"column": col, "variable": var, "scenario": scen, "unit": unit})

meta = pd.DataFrame(records)
scenarios = sorted({r["scenario"] for r in records if r["scenario"]})

variable_unit_pairs = []
for var, sub in meta.groupby("variable", dropna=True):
    units = {str(u).upper() for u in sub["unit"] if pd.notna(u)}
    for u in units:
        variable_unit_pairs.append((var, u))

variables = sorted([f"{v}__{u}" for v, u in variable_unit_pairs])

variable_labels = [
    {"label": f"{v} ({u})", "value": f"{v}__{u}"}
    for v, u in variable_unit_pairs
]

def add_water_year_column(df_in):
    df_copy = df_in.copy().sort_index()
    df_copy["Date"] = pd.to_datetime(df_copy.index)
    df_copy["Year"] = df_copy["Date"].dt.year
    df_copy["Month"] = df_copy["Date"].dt.month
    df_copy["WaterYear"] = np.where(df_copy["Month"] >= 10, df_copy["Year"] + 1, df_copy["Year"])
    return df_copy.drop(["Date", "Year", "Month"], axis=1)

water_year_df = add_water_year_column(df)

years_in_data = sorted(df.index.year.unique())
drought_years = {
    1924, 1925, 1926, 1929, 1930, 1931, 1932, 1933, 1934, 1939,
    1944, 1945, 1947, 1948, 1949, 1950, 1955, 1960, 1961, 1962, 1964,
    1976, 1977, 1979, 1981, 1987, 1988, 1989, 1990, 1991, 1992, 1994,
    2001, 2008, 2009, 2013, 2014, 2015, 2020, 2021
}
year_options = [{"label": f"{y} {'(Historical Drought Years)' if y in drought_years else ''}", "value": y} for y in years_in_data]
water_year_type_options = [{"label": str(i), "value": i} for i in range(1, 6)]
month_options = [{"label": name, "value": i} for i, name in enumerate(
    ["January","February","March","April","May","June","July","August","September","October","November","December"], start=1
)]

plot_type_descriptions = {
    "time_series": "Shows how the selected variable changes over time for each scenario.",
    "monthly": "Displays the monthly average across selected year(s).",
    "single_exceedance": "Shows the probability that a value will be equaled or exceeded.",
    "annual_exceedance": "Shows how often the selected month's total value exceeds a given threshold across all years.",
    "month_of_year_avg": "Averages each calendar month across all years, optionally filtered by Water Year Type."
}

def find_col(df_in, var_unit, scenario):
    var, unit = var_unit.split("__")
    suffix = f"_{unit.lower()}"
    matches = [
        c for c in df_in.columns
        if var in str(c)
        and scenario in str(c)
        and str(c).lower().endswith(suffix)
    ]
    return matches[0] if matches else None

def find_wyt_col(df_in, scenario):
    matches = [c for c in df_in.columns if f"CALSIM_WYT_SAC__{scenario}" in str(c) and "WATERYEARTYPE" in str(c)]
    return matches[0] if matches else None

def get_colors(scenarios_list):
    base = ["red", "blue", "green", "orange", "purple", "brown", "cyan", "magenta", "gray", "black"]
    return {s: base[i % len(base)] for i, s in enumerate(scenarios_list)}

def get_line_styles():
    return ["solid", "dash", "dot", "dashdot", "longdash", "longdashdot"]

def filter_by_wyt_annual(df_col, scenario, wyt_list, month=5):
    if not wyt_list:
        return df_col
    wyt_col = find_wyt_col(df, scenario)
    if wyt_col is None:
        return df_col
    working_df = df[[wyt_col]].copy()
    working_df["WaterYear"] = water_year_df["WaterYear"]
    working_df["Month"] = df.index.month
    filtered = working_df[working_df["Month"] == month].groupby("WaterYear").first()
    selected_years = filtered[filtered[wyt_col].isin(wyt_list)].index
    out = df_col.copy()
    out["WaterYear"] = water_year_df["WaterYear"]
    out = out[out["WaterYear"].isin(selected_years)]
    return out.drop(columns="WaterYear")

def normalize_scenario_token(tok):
    if tok is None:
        return None
    s = str(tok).strip()
    if not s:
        return None
    m = re.match(r"^s(\d+)$", s, flags=re.IGNORECASE)
    if m:
        n = int(m.group(1))
        return f"s{n:04d}"
    if re.match(r"^\d+$", s):
        n = int(s)
        return f"s{n:04d}"
    m2 = re.search(r"(s\d+)", s, flags=re.IGNORECASE)
    if m2:
        return normalize_scenario_token(m2.group(1))
    return None

def parse_unit_from_group_name(text):
    m = re.search(r"\(([^)]+)\)", str(text) if text is not None else "")
    return m.group(1).strip().upper() if m else None

def split_tokens(text):
    if text is None:
        return []
    s = str(text)
    parts = re.split(r"[,\n;|]+", s)
    return [p.strip() for p in parts if p and p.strip()]

def clean_var_token(token):
    if token is None:
        return None
    t = str(token).strip()
    if not t:
        return None
    t = re.sub(r"\s*\([^)]*\)\s*$", "", t).strip()
    return t if t else None

def unit_in_token(token):
    if token is None:
        return None
    m = re.search(r"\(([^)]+)\)\s*$", str(token).strip())
    return m.group(1).strip().upper() if m else None

def build_variable_groups(df_groups):
    colmap = {str(c).strip().lower(): c for c in df_groups.columns}
    c_group = colmap.get("grouping")
    c_desc = colmap.get("description")
    c_vars = colmap.get("variables")
    if c_group is None or c_vars is None:
        return {}
    groups = {}
    for _, row in df_groups.iterrows():
        name = str(row.get(c_group, "")).strip()
        if not name:
            continue
        desc = str(row.get(c_desc, "")).strip() if c_desc else ""
        unit = parse_unit_from_group_name(name)
        vars_raw = row.get(c_vars, "")
        tokens = split_tokens(vars_raw)
        cleaned = []
        for tok in tokens:
            v = clean_var_token(tok)
            if v:
                cleaned.append(tok.strip())
        groups[name] = {"description": desc, "unit": unit, "variables": cleaned}
    return groups

def build_scenario_groups(df_groups):
    colmap = {str(c).strip().lower(): c for c in df_groups.columns}
    c_group = colmap.get("grouping")
    c_desc = colmap.get("description")
    c_sc = colmap.get("scenarios")
    if c_group is None or c_sc is None:
        return {}
    groups = {}
    for _, row in df_groups.iterrows():
        name = str(row.get(c_group, "")).strip()
        if not name:
            continue
        desc = str(row.get(c_desc, "")).strip() if c_desc else ""
        sc_raw = row.get(c_sc, "")
        tokens = split_tokens(sc_raw)
        norm = []
        for tok in tokens:
            ns = normalize_scenario_token(tok)
            if ns:
                norm.append(ns)
        groups[name] = {"description": desc, "scenarios": norm}
    return groups

var_groups_store_init = build_variable_groups(var_groups_raw)
scen_groups_store_init = build_scenario_groups(scen_groups_raw)

available_varunit_values = {f"{v}__{u.upper()}" for v, u in variable_unit_pairs}

def group_options(store):
    return [{"label": k, "value": k} for k in sorted(store.keys())]

app = dash.Dash(__name__)

app.layout = html.Div([
    html.H2("Water Data Dashboard", style={"textAlign": "center"}),

    dcc.Store(id="var-groups-store", data=var_groups_store_init),
    dcc.Store(id="scen-groups-store", data=scen_groups_store_init),

    html.Div([
        html.Div([
            html.Label("Variable Groups"),
            dcc.Dropdown(
                id="var-group-dropdown",
                options=group_options(var_groups_store_init),
                value=None,
                multi=False
            ),
            html.Div(id="var-group-desc", style={"marginTop": "6px", "fontStyle": "italic", "color": "#555"}),
            html.Button("Add Variable Group to Selection", id="add-var-group-btn", n_clicks=0, style={"marginTop": "8px"}),
        ], style={"width": "48%", "display": "inline-block", "verticalAlign": "top"}),

        html.Div([
            html.Label("Scenario Groups"),
            dcc.Dropdown(
                id="scen-group-dropdown",
                options=group_options(scen_groups_store_init),
                value=None,
                multi=False
            ),
            html.Div(id="scen-group-desc", style={"marginTop": "6px", "fontStyle": "italic", "color": "#555"}),
            html.Button("Add Scenario Group to Selection", id="add-scen-group-btn", n_clicks=0, style={"marginTop": "8px"}),
        ], style={"width": "48%", "display": "inline-block", "marginLeft": "4%", "verticalAlign": "top"})
    ], style={"marginBottom": "14px"}),

    html.Label("Select Variables (must have same unit)"),
    dcc.Dropdown(
        id="variable-dropdown",
        options=variable_labels,
        value=[],
        multi=True
    ),

    html.Label("Select Scenarios"),
    dcc.Dropdown(
        id="scenario-dropdown",
        options=[{"label": s, "value": s} for s in scenarios],
        value=[],
        multi=True
    ),

    html.Label("Select Plot Type"),
    dcc.Dropdown(id="plot-type-dropdown", options=[
        {"label": "Time Series", "value": "time_series"},
        {"label": "Monthly-of-Year", "value": "monthly"},
        {"label": "Month-of-Year Avg", "value": "month_of_year_avg"},
        {"label": "Single Exceedance", "value": "single_exceedance"},
        {"label": "Annual Exceedance", "value": "annual_exceedance"}
    ], value="time_series"),

    html.Div(id="plot-type-description", style={"marginTop": "10px", "fontStyle": "italic", "color": "#555"}),

    html.Div([
        html.Label("Select Year(s) (for Monthly Plot)"),
        dcc.Dropdown(id="year-dropdown", options=year_options, value=[], multi=True)
    ], id="year-container", style={"display": "none"}),

    html.Div([
        html.Label("Select Water Year Type (1-5)"),
        dcc.Dropdown(id="wyt-dropdown", options=water_year_type_options, value=[], multi=True)
    ], id="wyt-container", style={"display": "none"}),

    html.Div([
        html.Label("Select Month for Annual Exceedance"),
        dcc.Dropdown(id="month-dropdown", options=month_options, value=9)
    ], id="month-container", style={"display": "none"}),

    dcc.Graph(id="dynamic-plot")
])

@app.callback(
    Output("var-group-desc", "children"),
    [Input("var-group-dropdown", "value"),
     Input("var-groups-store", "data")]
)
def show_var_group_desc(group_name, store):
    if not group_name or not store or group_name not in store:
        return ""
    unit = store[group_name].get("unit")
    desc = store[group_name].get("description", "")
    return f"{desc}  Unit: {unit}" if unit else desc

@app.callback(
    Output("scen-group-desc", "children"),
    [Input("scen-group-dropdown", "value"),
     Input("scen-groups-store", "data")]
)
def show_scen_group_desc(group_name, store):
    if not group_name or not store or group_name not in store:
        return ""
    desc = store[group_name].get("description", "")
    scs = store[group_name].get("scenarios", []) or []
    scs_avail = [s for s in scs if s in scenarios]
    return f"{desc}  Available in dashboard: {len(scs_avail)}"

@app.callback(
    Output("variable-dropdown", "value"),
    Input("add-var-group-btn", "n_clicks"),
    State("var-group-dropdown", "value"),
    State("var-groups-store", "data"),
    State("variable-dropdown", "value"),
    prevent_initial_call=True
)
def add_var_group_to_selection(_, group_name, store, current_selected):
    if not group_name or not store or group_name not in store:
        raise PreventUpdate

    current_selected = list(current_selected or [])
    group_unit = (store[group_name].get("unit") or "").upper() or None
    vars_list = store[group_name].get("variables", []) or []

    add_vals = []
    for raw_tok in vars_list:
        tok_unit = unit_in_token(raw_tok)
        vname = clean_var_token(raw_tok)
        if not vname:
            continue

        use_unit = tok_unit or group_unit
        if not use_unit:
            continue

        cand = f"{vname}__{use_unit}"
        if cand in available_varunit_values:
            add_vals.append(cand)

    merged = list(dict.fromkeys(current_selected + add_vals))
    return merged

@app.callback(
    Output("scenario-dropdown", "value"),
    Input("add-scen-group-btn", "n_clicks"),
    State("scen-group-dropdown", "value"),
    State("scen-groups-store", "data"),
    State("scenario-dropdown", "value"),
    prevent_initial_call=True
)
def add_scen_group_to_selection(_, group_name, store, current_selected):
    if not group_name or not store or group_name not in store:
        raise PreventUpdate
    current_selected = list(current_selected or [])
    group_scen = store[group_name].get("scenarios", []) or []
    group_scen_avail = [s for s in group_scen if s in scenarios]
    merged = list(dict.fromkeys(current_selected + group_scen_avail))
    return merged

@app.callback(
    [Output("dynamic-plot", "figure"),
     Output("year-container", "style"),
     Output("wyt-container", "style"),
     Output("month-container", "style"),
     Output("plot-type-description", "children")],
    [Input("variable-dropdown", "value"),
     Input("scenario-dropdown", "value"),
     Input("plot-type-dropdown", "value"),
     Input("year-dropdown", "value"),
     Input("wyt-dropdown", "value"),
     Input("month-dropdown", "value")]
)
def update_plot(vars_selected, scenarios_selected, plot_type, years_selected, wyt, month):
    show_year = {"display": "block"} if plot_type == "monthly" else {"display": "none"}
    show_wyt = {"display": "block"} if plot_type == "month_of_year_avg" else {"display": "none"}
    show_month = {"display": "block"} if plot_type == "annual_exceedance" else {"display": "none"}
    description = plot_type_descriptions.get(plot_type, "")

    if not scenarios_selected or not vars_selected:
        return go.Figure(), show_year, show_wyt, show_month, description

    selected_units = {v.split("__")[1] for v in vars_selected}
    if len(selected_units) > 1:
        fig = go.Figure()
        fig.update_layout(
            title="Unit Mismatch Detected",
            annotations=[{
                "text": "Selected variables have different units. Please select variables with the same unit.",
                "xref": "paper", "yref": "paper",
                "showarrow": False, "font": {"size": 16}
            }]
        )
        return fig, show_year, show_wyt, show_month, description

    fig = go.Figure()
    line_styles = get_line_styles()
    colors = get_colors(scenarios_selected)

    years_selected = list(years_selected or [])

    for idx, var_unit in enumerate(vars_selected):
        var, unit = var_unit.split("__")
        style = line_styles[idx % len(line_styles)]
        for s in scenarios_selected:
            col = find_col(df, var_unit, s)
            if not col:
                continue
            df_copy = df[[col]].copy()

            if plot_type == "monthly":
                df_copy["Year"] = df_copy.index.year
                df_copy["Month"] = df_copy.index.month
                df_sel = df_copy[df_copy["Year"].isin(years_selected)] if years_selected else df_copy.iloc[0:0]
                monthly_avg = df_sel.groupby("Month")[col].mean()
                fig.add_trace(go.Scatter(
                    x=monthly_avg.index, y=monthly_avg.values,
                    mode="lines+markers", name=f"{s} - {var} ({unit})",
                    line=dict(color=colors[s], dash=style)
                ))
            elif plot_type == "month_of_year_avg":
                df_sel = water_year_df[[col]].copy()
                df_sel = filter_by_wyt_annual(df_sel, s, wyt, month=5)
                df_sel["Month"] = df_sel.index.month
                monthly_avg = df_sel.groupby("Month")[col].mean()
                fig.add_trace(go.Scatter(
                    x=monthly_avg.index, y=monthly_avg.values,
                    mode="lines", name=f"{s} - {var} ({unit})",
                    line=dict(color=colors[s], dash=style)
                ))
            elif plot_type == "single_exceedance":
                series = df_copy[col].dropna().sort_values(ascending=False)
                exceedance_probs = np.arange(1, len(series) + 1) / (len(series) + 1)
                fig.add_trace(go.Scatter(
                    x=exceedance_probs, y=series.values,
                    mode="lines", name=f"{s} - {var} ({unit})",
                    line=dict(color=colors[s], dash=style)
                ))
            elif plot_type == "annual_exceedance":
                df_sel = df_copy[df_copy.index.month == month]
                annual_sum = df_sel.resample("YE").sum(min_count=1)
                sorted_vals = annual_sum[col].dropna().sort_values(ascending=False)
                exceed_probs = sorted_vals.rank(method="first", ascending=False) / (1 + len(sorted_vals))
                fig.add_trace(go.Scatter(
                    x=exceed_probs, y=sorted_vals,
                    mode="lines", name=f"{s} - {var} ({unit})",
                    line=dict(color=colors[s], dash=style)
                ))
            else:
                fig.add_trace(go.Scatter(
                    x=df.index, y=df_copy[col],
                    mode="lines", name=f"{s} - {var} ({unit})",
                    line=dict(color=colors[s], dash=style)
                ))

    unit_for_axis = selected_units.pop()
    fig.update_layout(
        title=f"{plot_type.replace('_', ' ').title()} Plot for Selected Variables",
        xaxis_title="Date" if plot_type in ["time_series", "monthly", "month_of_year_avg"] else "Exceedance Probability",
        yaxis_title=f"Value ({unit_for_axis})"
    )
    return fig, show_year, show_wyt, show_month, description

if __name__ == "__main__":
    app.run(debug=True, port=8040)


### Save data

In [ ]:
df.to_csv(output_filename)
df.to_csv(output_group_data_filename)
print("Data saved in " + output_filename + " and " + output_group_data_filename)